# 方針
## 特徴量を絞り込むことで正解率の向上を図る(id, day, monthを除く)
## 欠損値を中央値で穴埋めする(duration)
## 文字列型特徴量を新しい特徴量に変換することで正解率の向上を図る

In [ ]:
import pandas as pd
import matplotlib as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
file = 'Bank.csv'
df = pd.read_csv(file)
print(f'Head contents of {file} is \n{df.head(5)}\n')
columns = df.columns
print(f'Label of {file} : {columns}\n')
print(f'Matrix size : {df.shape}')

In [ ]:
# ラベルの意味の辞書を作成
label_means = { 
'id':'顧客ID',
'age':'年齢',
'job':'職種',
'marital':'既婚/未婚/離別など',
'education':'最終学歴',
'default':'債務不履行の有無',
'amount':'年間キャンペーン終了時点での総投資信託購入額',
'housing':'住宅ローンの有無',
'loan':'個人ローンの有無',
'contact':'連絡方法',
'day':'最終接触日',
'month':'最終接触月',
'duration':'接触時の平均時間(秒)',
'campaign':'現キャンペーン内での接触回数',
'previous':'キャンペーン前に接触した回数',
'y':'今回のキャンペーンの結果(1:購入、0:未購入)'
 }
print('ラベルの意味：')
for key, value in label_means.items():
    print(key,':',value)


In [ ]:
# 文字型特徴量のユニークな値を調べる
import matplotlib.pyplot as plt
%matplotlib inline

# for col in df.columns:
#    print(f'Unique item of {col} = {df[col].unique()}\n')
# i = 1; print(f'Unique item of {df.columns[i]} = {df[i].unique()}\n')
strlabels =  ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'y']
for strlabel in strlabels:
    print(f'Mean of {strlabel}:{label_means[strlabel]}')
    print(f'Unique item of {strlabel} = {df[strlabel].unique()}')
    value_counts = df[strlabel].value_counts()
    print(f'Each count of unique item of {strlabel}:{value_counts}')

    value_counts.plot(kind='bar')
    plt.show()
    # plt.close()

In [ ]:
# 文字列型特徴量の度数分布図をyの0/1で色分けする
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

strlabels =  ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month']
# strlabels =  ['job']
for strlabel in strlabels:
    print(f'Mean of {strlabel}:{label_means[strlabel]}')
    # ビン分け
    #bins = np.histogram_bin_edges(df[label], bins=10)
    # df['bin'] = pd.cut(df[label], bins=bins); # print(df.head(5))

    # 各ビンごとのlabel=1とlabel=0の数を集計
    bin_label_counts = df.groupby([strlabel, 'y']).size().unstack(fill_value=0)
    print('\n',bin_label_counts)

    # 棒グラフの描画
    x = np.arange(len(bin_label_counts)); #print(f'x={x}')
    width = 0.8

    fig, ax = plt.subplots()
    ax.bar(x, bin_label_counts[0], width, label='y=0', color='skyblue')
    ax.bar(x, bin_label_counts[1], width, bottom=bin_label_counts[0], label='y=1', color='salmon')

    # 軸とラベル
    ax.set_xticks(x)
    ax.set_xticklabels(df[strlabel].unique(), rotation=45)
    ax.set_xlabel(f'{strlabel}')
    ax.set_ylabel('Count')
    # ax.set_yscale('log')
    ax.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# 文字列型特徴量の度数分布図とyの0/1の割合の折れ線グラフの重ね合わせ表示
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

strlabels =  ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month']
# strlabels =  ['job']
for strlabel in strlabels:
    print(f'Mean of {strlabel}:{label_means[strlabel]}')
    # ビン分け
    #bins = np.histogram_bin_edges(df[label], bins=10)
    # df['bin'] = pd.cut(df[label], bins=bins); # print(df.head(5))

    # 各ビンごとのlabel=1とlabel=0の数を集計
    # bin_label_counts = df.groupby(['bin', 'y']).size().unstack(fill_value=0); print('\n',bin_label_counts)
    bin_counts = df.groupby(strlabel)['y'].count()
    bin_positives = df.groupby(strlabel)['y'].sum()
    bin_positive_rate = bin_positives / bin_counts

    # ヒストグラムと割合の棒グラフを重ねて表示
    fig, ax1 = plt.subplots()

    # ヒストグラム（全体の件数）
    ax1.bar(range(len(bin_counts)), bin_counts, width=0.4, label='Count', align='center', alpha=0.6)
    ax1.set_ylabel('Count')
    ax1.set_xlabel(f'{strlabel}')
    ax1.set_xticks(range(len(bin_counts)))
    ax1.set_xticklabels(df[strlabel].unique(), rotation=45)

    # 割合の折れ線グラフ
    ax2 = ax1.twinx()
    ax2.plot(range(len(bin_positive_rate)), bin_positive_rate, color='red', marker='o', label='Positive Rate (label=1)')
    ax2.set_ylabel('Positive Rate')

    # 凡例とレイアウト調整
    fig.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
# 数値型特徴量の度数分布図
import matplotlib.pyplot as plt
%matplotlib inline

labels = [ 'age', 'amount', 'duration', 'campaign', 'previous',]
for label in labels:
    print(f'{label}:{label_means[label]}')
    print(f'{label}の平均値：{df[label].mean():.1f}')
    print(f'{label}の中央値：{df[label].median():.1f}')
    print(f'{label}の標準偏差：{df[label].std():.1f}')
    df[label].plot(kind='hist',xlabel=label,logy=True)
    plt.show()
    plt.close()

In [ ]:
# 数値型特徴量の度数分布図をyの0/1で色分けする
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# labels = [ 'age', 'amount', 'duration', 'campaign', 'previous',]
labels = [ 'age', 'amount', 'campaign', 'previous']
for label in labels:
    print(f'{label}:{label_means[label]}')
    # ビン分け
    bins = np.histogram_bin_edges(df[label], bins=10)
    df['bin'] = pd.cut(df[label], bins=bins); # print(df.head(5))

    # 各ビンごとのlabel=1とlabel=0の数を集計
    bin_label_counts = df.groupby(['bin', 'y']).size().unstack(fill_value=0)
    print('\n',bin_label_counts)


    # 棒グラフの描画
    x = np.arange(len(bin_label_counts)); #print(f'x={x}')
    width = 0.8

    fig, ax = plt.subplots()
    ax.bar(x, bin_label_counts[0], width, label='y=0', color='skyblue')
    ax.bar(x, bin_label_counts[1], width, bottom=bin_label_counts[0], label='y=1', color='salmon')

    # 軸とラベル
    ax.set_xticks(x)
    ax.set_xticklabels([f'{interval.left:.1f}-{interval.right:.1f}' for interval in bin_label_counts.index], rotation=45)
    ax.set_xlabel(f'{label} Bins')
    ax.set_ylabel('Count')
    # ax.set_yscale('log')
    ax.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# 数値型特徴量の度数分布図とyの0/1の割合の折れ線グラフの重ね合わせ表示
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# labels = [ 'age', 'amount', 'duration', 'campaign', 'previous',]
labels = [ 'age', 'amount', 'campaign', 'previous']
for label in labels:
    print(f'{label}:{label_means[label]}')
    # ビン分け
    bins = np.histogram_bin_edges(df[label], bins=10)
    df['bin'] = pd.cut(df[label], bins=bins); # print(df.head(5))

    # 各ビンごとのlabel=1とlabel=0の数を集計
    # bin_label_counts = df.groupby(['bin', 'y']).size().unstack(fill_value=0); print('\n',bin_label_counts)
    bin_counts = df.groupby('bin')['y'].count()
    bin_positives = df.groupby('bin')['y'].sum()
    bin_positive_rate = bin_positives / bin_counts

    # ヒストグラムと割合の棒グラフを重ねて表示
    fig, ax1 = plt.subplots()

    # ヒストグラム（全体の件数）
    ax1.bar(range(len(bin_counts)), bin_counts, width=0.4, label='Count', align='center', alpha=0.6)
    ax1.set_ylabel('Count')
    ax1.set_xlabel(f'{label} Bins')
    ax1.set_xticks(range(len(bin_counts)))
    ax1.set_xticklabels([f'{interval.left:.1f}-{interval.right:.1f}' for interval in bin_counts.index], rotation=45)

    # 割合の折れ線グラフ
    ax2 = ax1.twinx()
    ax2.plot(range(len(bin_positive_rate)), bin_positive_rate, color='red', marker='o', label='Positive Rate (label=1)')
    ax2.set_ylabel('Positive Rate')

    # 凡例とレイアウト調整
    fig.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

In [ ]:
print('データのサイズの確認：',df.shape)
print('欠損値の数を確認：\n',df.isnull().sum())

In [ ]:
# 'duration'の欠損値をその列の中央値(外れ値があるため)で穴埋め
import numpy as np
label = 'duration'
df2 = pd.read_csv(file)
df2[label] = df[label].fillna(df[label].median())
print('欠損値の数を確認：\n',df2.isnull().sum())
df2.columns

In [ ]:
# 文字列型特徴量を新しい特徴量に変換する

labels_old_new = {'job':'income', 'marital':'divorced', 'contact':'contact_doc'}
new_class = [
#{'blue-collar':'middle', 'entrepreneur':'high', 'management':'high', 'retired':'low', 'services':'middle',
# 'technician':'high', 'admin.':'middle', 'self-employed':'high', 'housemaid':'low', 'unemployed':'low', 'unknown':'low', 'student':'low'},
{'blue-collar':'middle', 'entrepreneur':'high', 'management':'high', 'retired':'middle', 'services':'middle',
 'technician':'high', 'admin.':'middle', 'self-employed':'middle', 'housemaid':'low', 'unemployed':'middle', 'unknown':'middle', 'student':'middle'},
{'married':'no', 'single':'no', 'divorced':'yes'},
{'cellular':'no', 'telephone':'no', 'sending _document':'yes'}]
for i, (old, new) in enumerate(labels_old_new.items()):
    print(f'i={i}')
    for key, value in new_class[i].items():
        print(key,':',value)
        df2.loc[(df2[old] == key ) , new] = value
    # 古いラベル列を削除する
    df2 = df2.drop([old], axis=1)

print(df2.head(20))

In [ ]:
# get_dummies関数を用いて文字列を数値に変換する
# items = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact']
items = ['education', 'default', 'housing', 'loan', 'income', 'divorced', 'contact_doc']

for item in items:
    valued_item = pd.get_dummies(df2[item], drop_first = True, dtype = int,prefix=item);
    # print('Valued Items are :\n',valued_item.head(3))
    # ダミー変数を元のデータフレームに列接続する
    df2 = pd.concat([df2,valued_item], axis = 1)
    # print(df2.head(3))
    # 文字列特徴量の列を削除する
    df2 = df2.drop(item, axis=1)
    # print(df2.columns)
print(df2.columns)


In [ ]:
# 特徴量と正解データに分割する('id','day', 'month'を除く) 
"""
col = ['age', 'job', 'marital', 'education', 'default', 'amount',
       'housing', 'loan', 'contact', 'duration', 'campaign','previous', ]
x = df2[col]
"""
x = df2.drop(['id','day','month','y'], axis=1)
t = df2['y']
print('x=\n',x.columns)

In [ ]:
# ホールドアウト法により訓練・検証データとテストデータに分割する

# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
x_train_val, x_test, y_train_val, y_test = train_test_split(x,t,test_size=0.2, random_state=0)

print("訓練データおよび検証用データの特徴量の行列サイズ：",x_train_val.shape)
# print(x_train_val.head(3), "\n")
print("テストデータの特徴量の行列サイズ：",x_test.shape)
# print(x_test.head(3),"\n")

In [ ]:
# モデルチューニングの繰り返し作業のため学習用の関数を定義する
from sklearn import tree

# def learn(x, t, depth=3):
def learn(x, t, depth=5, weight = 'balanced'):
    x_train, x_test, y_train, y_test = train_test_split(x,t,test_size=0.2, random_state=0)

    #  criterion : gini=ジニ不純度(既定), entropy=情報利得), log_loss=ログ損失(分類の確率的評価))
    #　splitter ： best=最良の分割を探索, random=ランダムに候補を選び、その中で最良を選択（高速化向け）
    model = tree.DecisionTreeClassifier(max_depth=depth, 
            # random_state=0, class_weight = 'balanced' )
            random_state=0, class_weight = weight )
    model.fit(x_train, y_train)

    score = model.score(X = x_train, y = y_train)
    score2 = model.score(X = x_test, y = y_test)
    return round(score,3), round(score2,3), model

In [ ]:
# 木の深さによる正解率の変化を確認
df_score = pd.DataFrame(columns = ['tree_depth', 'train_score', 'test_score'])

for j in range(1,15):
    train_score, test_score, model = learn(x_train_val, y_train_val, depth = j)
    df_score.loc[len(df_score)]=[j,train_score,test_score]
    sentence = '訓練データの正解率{}'
    sentence2 = 'テストデータの正解率{}'
    total_sentence = '深さ{}：'+sentence+', '+sentence2
    # print(total_sentence.format(j, train_score, test_score))

print('df_score = \n', df_score)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(8,5))
plt.plot(df_score['tree_depth'], df_score['train_score'], marker='o')
plt.plot(df_score['tree_depth'], df_score['test_score'], marker='s')
plt.xlabel('tree_depth')
plt.ylabel('score')
plt.title('Line Plot of Score vs tree_depth')
plt.legend(loc='upper left', frameon=True)
# bbox_to_anchor=(1,1)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# 木の深さを10にして再学習してモデルを保存
s1, s2, model = learn(x_train_val, y_train_val, depth = 10)

# 学習モデルを保存
import pickle
with open('Bank_Test2.pkl','wb') as f:
    pickle.dump(model, f)

In [ ]:
df_feat = pd.DataFrame(model.feature_importances_, index = x_train_val.columns)
print('決定木分析における特徴量重要度を確認', df_feat)
df_feat.plot(kind='bar')
plt.show()

In [ ]:
# テストデータでのスコアを評価
model.score(x_test, y_test)